# Modelado Semántico Aplicado: Ontologías, RDF/OWL, SKOS y Knowledge Graphs

## Objetivos
- Construir una ontología mínima con **RDF/OWL** en Python (rdflib).
- Consultar con **SPARQL** para responder preguntas de negocio.
- Modelar una **taxonomía con SKOS** y relacionarla con datos.
- Mapear contratos de datos y columnas a **IRIs semánticos**.
- Sentar bases para **knowledge graphs** y semántica empresarial.

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Patrones de Arquitectura de Datos](08_patrones_arquitectura_datos.ipynb) | Siguiente: [Gobernanza de Metadatos Semánticos y Knowledge Graphs (Aplicado)](10_gobernanza_metadatos_semanticos_y_kg.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. Objetivos
2. 1. Introducción Rápida a Estándares
3. 2. Ontología mínima con rdflib
4. 3. Consultas SPARQL
5. 4. Taxonomías con SKOS
6. 5. Mapear Contratos de Datos a Ontologías
7. 6. Hacia Knowledge Graphs
8. 7. Ejercicio

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Modelado Semántico Aplicado: Ontologías, RDF/OWL, SKOS y Knowledge Graphs"] --> C["Notebook siguiente"]
```


## 1. Introducción Rápida a Estándares
- **RDF**: grafo de tripletas (sujeto, predicado, objeto).
- **RDFS/OWL**: vocabularios para clases, propiedades y restricciones.
- **SKOS**: estándar para taxonomías/tesauros (broader/narrower/related).
- **SPARQL**: lenguaje de consulta sobre grafos RDF.
- **IRI**: identificador global (ej: `https://example.com/ontology/Customer`).

## 2. Ontología mínima con rdflib
Definimos clases: `Customer`, `Order`, `Product`. Propiedades: `hasOrder`, `hasProduct`, `totalAmount`.

In [ ]:
try:
    from rdflib import Graph, Namespace, RDF, RDFS, OWL, Literal, XSD
    HAS_RDFLIB = True
except ModuleNotFoundError:
    HAS_RDFLIB = False

if HAS_RDFLIB:
    EX = Namespace('https://example.com/ontology/')
    g = Graph()
    g.bind('ex', EX)
    for cls in ['Customer', 'Order', 'Product']:
        g.add((EX[cls], RDF.type, OWL.Class))
    g.add((EX.hasOrder, RDF.type, OWL.ObjectProperty))
    g.add((EX.hasProduct, RDF.type, OWL.ObjectProperty))
    g.add((EX.totalAmount, RDF.type, OWL.DatatypeProperty))
    g.add((EX.hasOrder, RDFS.domain, EX.Customer))
    g.add((EX.hasOrder, RDFS.range, EX.Order))
    g.add((EX.hasProduct, RDFS.domain, EX.Order))
    g.add((EX.hasProduct, RDFS.range, EX.Product))
    cust1 = EX['customer/1001']
    ord1 = EX['order/5001']
    prod1 = EX['product/2007']
    g.add((cust1, RDF.type, EX.Customer))
    g.add((ord1, RDF.type, EX.Order))
    g.add((prod1, RDF.type, EX.Product))
    g.add((cust1, EX.hasOrder, ord1))
    g.add((ord1, EX.hasProduct, prod1))
    g.add((ord1, EX.totalAmount, Literal('1200.00', datatype=XSD.decimal)))
    print(g.serialize(format='turtle'))
else:
    conceptual_triples = [
        ('Customer', 'hasOrder', 'Order'),
        ('Order', 'hasProduct', 'Product'),
        ('Order', 'totalAmount', 'decimal')
    ]
    for triple in conceptual_triples:
        print(triple)


## 3. Consultas SPARQL
Ejemplo: obtener pedidos y montos por cliente.

In [ ]:
if HAS_RDFLIB:
    q = '''
    PREFIX ex: <https://example.com/ontology/>
    SELECT ?customer ?order ?amount
    WHERE {
      ?customer a ex:Customer ; ex:hasOrder ?order .
      ?order ex:totalAmount ?amount .
    }
    '''
    for row in g.query(q):
        print(row)
else:
    print(('customer/1001', 'order/5001', '1200.00'))


## 4. Taxonomías con SKOS
Creamos una taxonomía de producto y usamos `skos:broader`/`skos:narrower`.

In [ ]:
if HAS_RDFLIB:
    from rdflib.namespace import SKOS
    g.bind('skos', SKOS)
    electronics = EX['concept/electronics']
    phones = EX['concept/electronics/phones']
    smartphones = EX['concept/electronics/phones/smartphones']
    for concept, label in [(electronics, 'Electronics'), (phones, 'Phones'), (smartphones, 'Smartphones')]:
        g.add((concept, RDF.type, SKOS.Concept))
        g.add((concept, SKOS.prefLabel, Literal(label)))
    g.add((phones, SKOS.broader, electronics))
    g.add((smartphones, SKOS.broader, phones))
    q = '''
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    SELECT ?narrower ?label
    WHERE { ?narrower skos:broader <https://example.com/ontology/concept/electronics> ; skos:prefLabel ?label . }
    '''
    for row in g.query(q):
        print(row)
else:
    print('Electronics -> Phones -> Smartphones')


## 5. Mapear Contratos de Datos a Ontologías
Vinculamos columnas a IRIs (semántica explícita) para interoperabilidad.

In [ ]:
contract = {
    'dataset': 'orders_silver',
    'columns': {
        'customer_id': 'https://example.com/ontology/Customer',
        'order_id': 'https://example.com/ontology/Order',
        'total_amount': 'https://example.com/ontology/totalAmount'
    }
}

if HAS_RDFLIB:
    from rdflib import URIRef
    def validar_contrato(contract, graph):
        missing = []
        for col, iri in contract['columns'].items():
            ref = URIRef(iri)
            if (ref, None, None) not in graph and (None, None, ref) not in graph:
                missing.append((col, iri))
        return missing
    print(validar_contrato(contract, g))
else:
    print('Contrato revisado de forma conceptual:', contract['dataset'])


## 6. Hacia Knowledge Graphs
- Un **KG** conecta ontologías + datos operativos + metadatos.
- Casos: 360 de cliente, recomendaciones, cumplimiento (linaje semántico).
- Integración con catálogo y lineage aporta descubrimiento y trazabilidad.

## 7. Ejercicio
1) Añade clase `Channel` y relación `purchasedVia` a la ontología.
2) Crea 2 instancias de `Order` con distintos canales.
3) Escribe una consulta SPARQL para contar pedidos por canal.

<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- W3C RDF: https://www.w3.org/RDF/
- W3C SPARQL: https://www.w3.org/TR/sparql11-query/
- Referencias del curso: ../04-recursos/referencias.md
